# 05 — Aggregate Results Across Seeds

Reads `metrics.json` for every `(condition × seed)` run and computes mean ± std.
Run this after completing training + eval for all seeds of a given priority tier.

**Key metric for the paper:** Uncertain-class recall across PW scale — the core claim.

In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    OUTPUT_BASE = '/content/drive/MyDrive/slm_logic_hardening/outputs'
    %cd '/content/drive/MyDrive/slm_logic_hardening'
else:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve()))
    OUTPUT_BASE = 'outputs'

# ── Configuration ─────────────────────────────────────────────────────────────
SEEDS = [42, 123, 7]

# P0 conditions (must-have)
P0_CONDITIONS = [
    'phi35_lora_folio',
    'phi35_lora_folio_pw_2k',
    'phi35_lora_folio_pw_5k',
    'phi35_lora_folio_pw_10k',
]

# P1 conditions (insurance)
P1_CONDITIONS = [
    'phi35_lora_folio_pw_20k',
    'phi35_lora_folio_pw_2k_rebal',
]

# Change to 'P1' or 'all' to include P1 conditions
PRIORITY = 'P0'

if PRIORITY == 'P0':
    CONDITIONS = P0_CONDITIONS
elif PRIORITY == 'P1':
    CONDITIONS = P1_CONDITIONS
else:
    CONDITIONS = P0_CONDITIONS + P1_CONDITIONS

print(f'Aggregating {len(CONDITIONS)} conditions × {len(SEEDS)} seeds')
print('Conditions:', CONDITIONS)

In [ ]:
import json
from pathlib import Path

def load_metrics(output_base: str, condition: str, seed: int) -> dict | None:
    path = Path(output_base) / 'phi35' / condition / f'seed_{seed}' / 'metrics.json'
    if not path.exists():
        print(f'  MISSING: {path}')
        return None
    with open(path) as f:
        return json.load(f)

# Load all available metrics
raw: dict[str, list[dict]] = {}
for cond in CONDITIONS:
    raw[cond] = []
    for seed in SEEDS:
        m = load_metrics(OUTPUT_BASE, cond, seed)
        if m is not None:
            m['_seed'] = seed
            raw[cond].append(m)
    found = len(raw[cond])
    print(f'{cond}: {found}/{len(SEEDS)} seeds found')

In [ ]:
import numpy as np

def mean_std(values: list[float]) -> tuple[float, float]:
    if not values:
        return float('nan'), float('nan')
    return float(np.mean(values)), float(np.std(values, ddof=0))

def extract_metrics(m: dict) -> dict:
    cr = m.get('classification_report', {})
    return {
        'accuracy':          m.get('accuracy', float('nan')),
        'true_f1':           cr.get('True',      {}).get('f1-score',  float('nan')),
        'false_f1':          cr.get('False',     {}).get('f1-score',  float('nan')),
        'uncertain_f1':      cr.get('Uncertain', {}).get('f1-score',  float('nan')),
        'true_recall':       cr.get('True',      {}).get('recall',    float('nan')),
        'false_recall':      cr.get('False',     {}).get('recall',    float('nan')),
        'uncertain_recall':  cr.get('Uncertain', {}).get('recall',    float('nan')),
        'true_prec':         cr.get('True',      {}).get('precision', float('nan')),
        'false_prec':        cr.get('False',     {}).get('precision', float('nan')),
        'uncertain_prec':    cr.get('Uncertain', {}).get('precision', float('nan')),
        'macro_f1':          cr.get('macro avg', {}).get('f1-score',  float('nan')),
        'invalid_rate':      m.get('invalid_rate', float('nan')),
        'uncertain_pred_n':  m.get('prediction_distribution', {}).get('Uncertain', float('nan')),
    }

METRIC_KEYS = list(extract_metrics({}  if not raw else (raw[CONDITIONS[0]][0] if raw[CONDITIONS[0]] else {})).keys())

aggregated: dict[str, dict] = {}
for cond, entries in raw.items():
    if not entries:
        aggregated[cond] = {'seed_count': 0}
        continue
    per_metric: dict[str, list[float]] = {k: [] for k in METRIC_KEYS}
    per_seed_rows = []
    for m in entries:
        ex = extract_metrics(m)
        per_seed_rows.append({'seed': m['_seed'], **ex})
        for k, v in ex.items():
            per_metric[k].append(v)
    stats = {}
    for k, vals in per_metric.items():
        mu, sd = mean_std(vals)
        stats[f'{k}_mean'] = round(mu, 4)
        stats[f'{k}_std']  = round(sd, 4)
    aggregated[cond] = {'seed_count': len(entries), 'per_seed': per_seed_rows, **stats}

print('Aggregation complete.')

In [ ]:
import pandas as pd

# ── Summary table (paper-ready) ───────────────────────────────────────────────
DISPLAY_COLS = [
    ('accuracy_mean',       'Acc mean'),
    ('accuracy_std',        'Acc std'),
    ('uncertain_recall_mean','Unc-R mean'),
    ('uncertain_recall_std', 'Unc-R std'),
    ('uncertain_f1_mean',    'Unc-F1 mean'),
    ('uncertain_f1_std',     'Unc-F1 std'),
    ('true_f1_mean',         'True-F1 mean'),
    ('false_f1_mean',        'False-F1 mean'),
    ('macro_f1_mean',        'Macro-F1 mean'),
    ('invalid_rate_mean',    'Invalid% mean'),
    ('uncertain_pred_n_mean','Unc preds mean'),
    ('seed_count',           'N seeds'),
]

rows = []
for cond in CONDITIONS:
    agg = aggregated.get(cond, {})
    row = {'Condition': cond}
    for src_key, display_name in DISPLAY_COLS:
        row[display_name] = agg.get(src_key, float('nan'))
    rows.append(row)

df = pd.DataFrame(rows).set_index('Condition')
pd.set_option('display.float_format', lambda x: f'{x:.3f}' if not pd.isna(x) else 'nan')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df.to_string())

In [ ]:
import matplotlib.pyplot as plt

# ── Plot: Uncertain-class recall vs PW scale ──────────────────────────────────
scale_map = {
    'phi35_lora_folio':           0,
    'phi35_lora_folio_pw_2k':     2000,
    'phi35_lora_folio_pw_5k':     5000,
    'phi35_lora_folio_pw_10k':    10000,
    'phi35_lora_folio_pw_20k':    20000,
    'phi35_lora_folio_pw_2k_rebal': None,  # excluded from scale plot
}

scale_conds = [c for c in CONDITIONS if scale_map.get(c) is not None]
x_vals   = [scale_map[c] for c in scale_conds]
unc_mean = [aggregated.get(c, {}).get('uncertain_recall_mean', float('nan')) for c in scale_conds]
unc_std  = [aggregated.get(c, {}).get('uncertain_recall_std',  0.0)          for c in scale_conds]
acc_mean = [aggregated.get(c, {}).get('accuracy_mean',         float('nan')) for c in scale_conds]
acc_std  = [aggregated.get(c, {}).get('accuracy_std',          0.0)          for c in scale_conds]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].errorbar(x_vals, unc_mean, yerr=unc_std, marker='o', capsize=4, color='#e74c3c')
axes[0].set_title('Uncertain-class Recall vs PW Scale')
axes[0].set_xlabel('ProofWriter training examples')
axes[0].set_ylabel('Recall (mean ± std, 3 seeds)')
axes[0].set_xscale('symlog', linthresh=1000)
axes[0].grid(True, alpha=0.3)

axes[1].errorbar(x_vals, acc_mean, yerr=acc_std, marker='o', capsize=4, color='#3498db')
axes[1].set_title('Overall Accuracy vs PW Scale')
axes[1].set_xlabel('ProofWriter training examples')
axes[1].set_ylabel('Accuracy % (mean ± std, 3 seeds)')
axes[1].set_xscale('symlog', linthresh=1000)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{OUTPUT_BASE}/uncertain_recall_vs_pw_scale.png', dpi=150)
plt.show()
print('Figure saved.')

In [ ]:
# ── Per-seed breakdown table ──────────────────────────────────────────────────
per_seed_rows = []
for cond in CONDITIONS:
    for row in aggregated.get(cond, {}).get('per_seed', []):
        per_seed_rows.append({'condition': cond, **row})

if per_seed_rows:
    df_seeds = pd.DataFrame(per_seed_rows).set_index(['condition', 'seed'])
    cols_show = ['accuracy', 'uncertain_recall', 'uncertain_f1', 'true_f1', 'false_f1', 'macro_f1']
    print(df_seeds[[c for c in cols_show if c in df_seeds.columns]].to_string())

In [ ]:
import csv

# ── Save CSV and JSON ─────────────────────────────────────────────────────────
csv_path  = Path(OUTPUT_BASE) / 'aggregate_results.csv'
json_path = Path(OUTPUT_BASE) / 'aggregate_results.json'

df.reset_index().to_csv(csv_path, index=False)

with open(json_path, 'w') as f:
    json.dump(aggregated, f, indent=2)

print(f'CSV  saved: {csv_path}')
print(f'JSON saved: {json_path}')